# 16. Ponto de entrada `python -m app`

Desenvolve `_dados_demo`, `_comando_ingestao`, `_analisar` e `main`. **F14, NF2, NF5.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import argparse
import numpy as np
import pandas as pd
import os
from app.principal import executar_pipeline

## Desenvolvimento

As funções abaixo foram escritas aqui e, após os testes, movidas para `app/__main__.py`.

In [2]:
# Periodos por ano, banco, n_scenarios e a serie sintetica de cada frequencia.
# A mensal vem primeiro por ser o padrao.

PERFIS = {
    "1mo": {"periodos_por_ano": 12,  "db": os.path.join("data", "mercado.db"),
            "n_scenarios": 200_000, "unidade": "mes",
            "mu": 0.015, "sigma": 0.06, "cdi": 0.008, "freq_pandas": "MS"},
    "1d":  {"periodos_por_ano": 252, "db": os.path.join("data", "mercado_diario.db"),
            "n_scenarios": 4_000_000, "unidade": "pregao",
            "mu": 0.0007, "sigma": 0.011, "cdi": 0.00049, "freq_pandas": "B"},
}

BETA_ANUAL = 0.96
GAMMA = 5.0
ANOS = 5
W0 = 1.0

In [3]:
def _dados_demo(perfil: dict, n: int = 1_050, seed: int = 7) -> pd.DataFrame:
    """Serie inventada de retornos, na frequencia do perfil escolhido."""

    rng = np.random.default_rng(seed)
    ruido = rng.normal(0.0, perfil["sigma"], n)
    ruido -= ruido.mean() # media exatamente 0
    datas = pd.date_range("2022-05-24", periods=n, freq=perfil["freq_pandas"])
    fmt = "%Y-%m-%d" if perfil["periodos_por_ano"] == 252 else "%Y-%m"
    return pd.DataFrame({
        "data": datas.strftime(fmt),
        "ibov": perfil["mu"] + ruido,
        "cdi": np.full(n, perfil["cdi"]),
    })

In [ ]:
def _analisar(argv) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        prog="python -m app",
        description="Roda a esteira de Samuelson (1969) sobre Ibovespa + CDI.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    p.add_argument("--diario", action="store_true",
                   help="usa a base diaria (252 pregoes) no lugar da mensal")
    p.add_argument("--anos", type=float, default=ANOS,
                   help="horizonte de planejamento T, em anos")
    p.add_argument("--beta-anual", type=float, default=BETA_ANUAL,
                   help="fator de desconto ANUAL (convertido para o periodo)")
    p.add_argument("--gamma", type=float, default=GAMMA,
                   help="coeficiente de aversao relativa ao risco")
    p.add_argument("--cdi-anual", type=float, default=None,
                   help="CDI ANUAL em decimal (0.13 = 13%% a.a.), convertido "
                        "para o periodo; se omitido, usa a media da serie de "
                        "CDI dos dados")
    p.add_argument("--w0", type=float, default=W0, help="riqueza inicial")
    p.add_argument("--n-scenarios", type=int, default=0,
                   help="cenarios de Monte Carlo; 0 = automatico por frequencia "
                        "(200k mensal, 4M diario)")
    p.add_argument("--n-paths", type=int, default=3_000,
                   help="trajetorias simuladas na propagacao pra frente")
    p.add_argument("--seed", type=int, default=1, help="semente (reprodutibilidade)")
    p.add_argument("--graficos", action="store_true",
                   help="alem de imprimir, escreve as figuras em results/")
    args = p.parse_args(argv) 
    if args.cdi_anual is not None and not -0.5 < args.cdi_anual < 1.0:
        p.error(f"--cdi-anual e em decimal e deve ficar entre -0.5 e 1.0; veio "
                f"{args.cdi_anual:g}. Para 13% ao ano use 0.13, e nao 13.")
    if args.anos <= 0:
        p.error(f"--anos deve ser positivo; veio {args.anos:g}.")
    return args

In [5]:
def _comando_ingestao(periodos_por_ano: int) -> str:
    """A linha de ingestao que enche o banco da frequencia pedida."""
    if periodos_por_ano == 252:
        return "python -m app.ingestao 2022-05-22 --diario"
    return "python -m app.ingestao"

In [ ]:
def main(argv=()) -> None:
    """Executa a esteira com os parametros de ``argv`` e imprime o resultado."""
    args = _analisar(list(argv))
    perfil = PERFIS["1d" if args.diario else "1mo"]
    ppa = perfil["periodos_por_ano"]
    unid = perfil["unidade"]
    T = int(round(ppa * args.anos))
    n_scenarios = args.n_scenarios or perfil["n_scenarios"]

    comum = {"ativos": ["ibov"], "periodos_por_ano": ppa, "gamma": args.gamma,
             "beta_anual": args.beta_anual, "w0": args.w0,
             "horizonte": T, "n_scenarios": n_scenarios,
             "n_paths": args.n_paths, "seed": args.seed}
    if args.cdi_anual is not None:
        comum["cdi_anual"] = args.cdi_anual

    origem_rf = ("do --cdi-anual" if args.cdi_anual is not None
                 else "da serie de CDI dos dados")
    dados_reais = os.path.exists(perfil["db"])
    if dados_reais:
        print(f"(dados REAIS: {perfil['db']} - R_f vem {origem_rf})")
        config = {"db_path": perfil["db"], "tabela": "retornos", **comum}
    else:
        print(f"(SEM banco real -> dados SINTETICOS de demonstracao; "
              f"R_f vem {origem_rf})")
        print(f"  para baixar dados reais:  {_comando_ingestao(ppa)}")
        config = {"retornos": _dados_demo(perfil), **comum}
    res = executar_pipeline(config)

    rf_a = (1 + res["rf"]) ** ppa - 1
    mu_a = (1 + res["mu_hat"][0]) ** ppa - 1

    def linha(rotulo: str, valor: str) -> None: 
        """Mantem a coluna dos valores alinhada, com 'mes' ou com 'pregao'."""
        print(f"{rotulo:<22}: {valor}")

    print(f"=== Esteira DP-CRRA-IID (Samuelson 1969) - base "
          f"{'diaria' if args.diario else 'mensal'} ===")
    linha("Ativos de risco", f"{res['ativos']}")
    linha(f"R_f ({unid})", f"{res['rf']:.8f}   ({rf_a:.2%} a.a.)")
    linha(f"mu_hat ({unid})", f"{res['mu_hat'][0]:.8f}   ({mu_a:.2%} a.a.)")
    linha("gamma", f"{args.gamma}")
    linha("Carteira otima a*", f"{np.round(res['alpha_star'], 4)}")
    linha("Phi_hat", f"{res['phi_hat']:.6f}")
    linha("beta", f"{res['beta']:.6f} por {unid}  ({args.beta_anual:.4g} a.a.)")
    linha(f"theta_0 ({unid})", f"{res['theta'][0]:.6f}")
    linha("theta_T (terminal)", f"{res['theta'][-1]:.4f}")

    por_ano = res["consumo_por_ano"] 
    print(f"Consumo por ano (frac. de W_0), horizonte de {args.anos:g} anos:")
    for i, total in enumerate(por_ano, start=1): 
        parcial = " (ano parcial)" if i == len(por_ano) and T % ppa else "" 
        print(f"   ano {i}: {total:.4f}{parcial}")
    linha(f"E[W_T] (T={res['horizonte']})",
          f"{res['E_W_T']:.6f}  [P5={res['W_T_p5']:.6f}, P95={res['W_T_p95']:.6f}]")

    if args.graficos:
        from app import graficos
        from app.mercado import RendaVariavel
        ret = config.get("retornos")
        if ret is None:
            from app import dal
            ret = dal.ler_sqlite(config["db_path"], config["tabela"])
        mercado = RendaVariavel(ret[["data"] + config["ativos"]])
        rodape = graficos.montar_rodape(
            res, comum, (ret["data"].iloc[0], ret["data"].iloc[-1]), len(ret),
            args.beta_anual, args.anos, "diario" if ppa == 252 else "mensal",
            dados_reais=dados_reais)
        destino = (graficos.DESTINO_PADRAO if dados_reais
                   else os.path.join(graficos.DESTINO_PADRAO, "sinteticos"))
        escritos = graficos.gerar(res, mercado, res["rf"], comum, rodape,
                                  destino=destino)
        print(f"Figuras escritas em {destino}/:")
        for caminho in escritos:
            print(f"   {os.path.basename(caminho)}")

**Teste**: `main()` roda a esteira e imprime o resultado.

In [7]:
import io, contextlib

In [8]:
print(_dados_demo(PERFIS["1d"]).head())

         data      ibov      cdi
0  2022-05-24  0.001552  0.00049
1  2022-05-25  0.004825  0.00049
2  2022-05-26 -0.001477  0.00049
3  2022-05-27 -0.008258  0.00049
4  2022-05-30 -0.003463  0.00049


In [9]:
os.chdir(RAIZ)
buf = io.StringIO()

In [10]:
with contextlib.redirect_stdout(buf): main()
print(buf.getvalue())

(dados REAIS: data\mercado.db - R_f vem da serie de CDI dos dados)
=== Esteira DP-CRRA-IID (Samuelson 1969) - base mensal ===
Ativos de risco       : ['ibov']
R_f (mes)             : 0.00737647   (9.22% a.a.)
mu_hat (mes)          : 0.01006999   (12.78% a.a.)
gamma                 : 5.0
Carteira otima a*     : [0.1232]
Phi_hat               : 0.970428
beta                  : 0.996604 por mes  (0.96 a.a.)
theta_0 (mes)         : 0.019895
theta_T (terminal)    : 1.0000
Consumo por ano (frac. de W_0), horizonte de 5 anos:
   ano 1: 0.2401
   ano 2: 0.2431
   ano 3: 0.2459
   ano 4: 0.2489
   ano 5: 0.2518
E[W_T] (T=60)         : 0.021107  [P5=0.019030, P95=0.023261]



In [11]:
saida_padrao = buf.getvalue()
assert 'Carteira otima' in saida_padrao
assert 'base mensal' in saida_padrao

**Teste**: as flags. O --cdi-anual tem default None e é recusado se vier em porcentagem; o --diario liga a base diária, e a antiga --mensal não existe mais.

In [12]:
sem_flag = _analisar([])
com_flag = _analisar(['--cdi-anual', '0.08'])

try:
    _analisar(['--cdi-anual', '13'])
    recusou_percentual = False
except SystemExit as e:
    recusou_percentual = (e.code == 2)

# a frequencia agora e escolhida por --diario; a mensal e o padrao
try:
    _analisar(['--mensal'])
    mensal_removido = False
except SystemExit as e:
    mensal_removido = (e.code == 2)

print('sem a flag:', sem_flag.cdi_anual, '| com a flag:', com_flag.cdi_anual)
print('recusou --cdi-anual 13?', recusou_percentual)
print('padrao e mensal?', not sem_flag.diario, '| --diario liga a diaria?', _analisar(['--diario']).diario)
print('--mensal foi removido?', mensal_removido)

sem a flag: None | com a flag: 0.08
recusou --cdi-anual 13? True
padrao e mensal? True | --diario liga a diaria? True
--mensal foi removido? True


usage: python -m app [-h] [--diario] [--anos ANOS] [--beta-anual BETA_ANUAL]
                     [--gamma GAMMA] [--cdi-anual CDI_ANUAL] [--w0 W0]
                     [--n-scenarios N_SCENARIOS] [--n-paths N_PATHS]
                     [--seed SEED] [--graficos]
python -m app: error: --cdi-anual e em decimal e deve ficar entre -0.5 e 1.0; veio 13. Para 13% ao ano use 0.13, e nao 13.
usage: python -m app [-h] [--diario] [--anos ANOS] [--beta-anual BETA_ANUAL]
                     [--gamma GAMMA] [--cdi-anual CDI_ANUAL] [--w0 W0]
                     [--n-scenarios N_SCENARIOS] [--n-paths N_PATHS]
                     [--seed SEED] [--graficos]
python -m app: error: unrecognized arguments: --mensal


In [13]:
assert sem_flag.cdi_anual is None
assert com_flag.cdi_anual == 0.08
assert recusou_percentual
assert sem_flag.diario is False and _analisar(['--diario']).diario is True
assert mensal_removido

**Teste**: com --cdi-anual, o R_f passa a vir da flag e não da série, e a conversão de ano para pregão é composta.

In [14]:
buf2 = io.StringIO()
with contextlib.redirect_stdout(buf2):
    main(['--diario', '--cdi-anual', '0.08', '--anos', '1',
          '--n-scenarios', '20000', '--n-paths', '200'])
saida_flag = buf2.getvalue()

print(saida_flag)

(dados REAIS: data\mercado_diario.db - R_f vem do --cdi-anual)
=== Esteira DP-CRRA-IID (Samuelson 1969) - base diaria ===
Ativos de risco       : ['ibov']
R_f (pregao)          : 0.00030545   (8.00% a.a.)
mu_hat (pregao)       : 0.00051104   (13.74% a.a.)
gamma                 : 5.0
Carteira otima a*     : [0.1505]
Phi_hat               : 0.998754
beta                  : 0.999838 por pregao  (0.96 a.a.)
theta_0 (pregao)      : 0.004095
theta_T (terminal)    : 1.0000
Consumo por ano (frac. de W_0), horizonte de 1 anos:
   ano 1: 1.0393
E[W_T] (T=252)        : 0.004155  [P5=0.003987, P95=0.004322]



In [15]:
assert 'R_f vem do --cdi-anual' in saida_flag
assert '(8.00% a.a.)' in saida_flag
assert '(13.12% a.a.)' not in saida_flag
assert 'dados REAIS' in saida_flag